In [3]:
import os
import sys

import collections
import xarray as xr
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

In [4]:
def longitude_flip(var):
    """ Convert longitude values from the -180:180 to 0:360 convention or vice versa.

        ** Only works for global data. Do not apply to data with a clipped longitude range **

        Parameters
        ----------
        var : Data Array
    """
    # get var info
    x,_=get_xy_coords(var) # extract original longitude values
    lon_name=x.name        # store name of longitude coordinate
    nx=len(x)              # longitude resolution

    # determine longitude format and create an array of new lons in opposite convention
    if min(x)<0:
        # if there are negative values, data is -180:180 and need to switch to 0:360
        new_lons=np.linspace((min(x)+180), (max(x)+180), nx)
    elif max(x)>180:
        # if the max value is >180, data is in 0:360 format and need to switch to -180:180
        new_lons=np.linspace((min(x)-180), (max(x)-180), nx)

    # shift the data by 180° of longitude
    nshift=nx//2
    var=var.roll({lon_name: nshift}, roll_coords=False)

    # update longitude coord with new values
    var=var.assign_coords({lon_name: new_lons})

    # add attributes documenting change
    timestamp=datetime.now().strftime("%B %d, %Y, %r")
    var.attrs['history']=f'flipped longitudes {timestamp}'
    var.attrs['original_lons']=x.values

    return(var)

def wind_stress_curl(taux,tauy):
    _, tauxdy = mp.geospatial_gradient(taux)
    tauydx, _ = mp.geospatial_gradient(tauy)
    return(tauydx - tauxdy)

**Eastward Surface Ocean Currents**

In [ ]:
## +++ LOAD & PROCESS DATA +++ ##

# set lat/lon bounds
xmin=190
xmax=260

# NASA GISS E2.1
uocn_e2pt1 = { 'ctrl':{}, '1sEnv.ca-only':{}, '1sEnv.ca-sa':{} }
for key in uocn_e2pt1.keys():
    dat = xr.open_dataset(f'aogcm.{key}.u_current.nc').u.squeeze(drop=True) * 0.01 # convert from m/s to cm/s
    dat = dat.rename({'lono':'lon', 'lato':'lat'}) # switch to standard dim names
    dat = longitude_flip(dat) # change longitude convetion to 0:360
    uocn_e2pt1[key] = dat.mean(dim='month').sel(lon=slice(xmin,xmax)) # calculate annual mean and along lon bounds
    
# Oscar obs
dat = xr.open_dataset('obs.oscar.u_current.nc').U.squeeze(drop=True).where(uobs>-1e32) # exclude missing values
dat = dat.rename({'LONGITUDE':'lon', 'LATITUDE':'lat'}) # switch to standard dim names
ucon_obs = dat.sel(lon=slice(xmin,xmax)) # already is annual mean
#uocn_obs_lr = uocn_obs.interp_like(uocn_e2pt1['ctrl'], method='linear') # interpolate to e2.1 resolution

# GFDL CM2.5-FLOR
uocn_flor = { 'ctrl':{}, 'hitopo':{} }
for key in uocn_flor.keys():
    dat = xr.open_dataset(f'flor.{key}.u.nc').u.squeeze(drop=True).groupby("time.month").mean()
    dat = dat.rename({'xu_ocean':'lon', 'yu_ocean':'lat'}) # switch to standard dim names
    uocn_flor[key] = dat.mean(dim='month').sel(lon=slice(xmin,xmax)) # calculate annual mean and along lon bounds
    

**Wind Stress Curl**

In [ ]:
## +++ LOAD DATA +++ ##
# need to load surface wind stress (tau) and calculate curl

# NASA GISS E2.1
taux_e2pt1={ 'ctrl':{}, '1sEnv.ca-sa':{}, '1sEnv.ca-only':{} }
tauy_e2pt1={ 'ctrl':{}, '1sEnv.ca-sa':{}, '1sEnv.ca-only':{} }
for key in e2pt1_taux.keys():
    taux_e2pt1[key] = xr.open_dataset(f'aogcm.{key}.taux.nc').taux.mean(dim='time')
    tauy_e2pt1[key] = xr.open_dataset(f'aogcm.{key}.tauy.nc').tauy.mean(dim='time')
    
# MERRA-2
taux_merra=xr.open_dataset(f'merra2.taux-tauy.nc').TAUXWTR.mean(dim='time')
tauy_merra=xr.open_dataset(f'merra2.taux-tauy.nc').TAUYWTR.mean(dim='time')

# GFDL CM2.5-FLOR
taux_flor={ 'ctrl':{}, 'hitopo':{} }
tauy_flor={ 'ctrl':{}, 'hitopo':{} }
for key in taux_flor.keys():
    taux_flor[key] = xr.open_dataset(f'flor.{key}.taux.nc').tau_x.mean(dim='time')
    tauy_flor[key] = xr.open_dataset(f'flor.{key}.taux.nc').tau_x.mean(dim='time')

In [ ]:
## +++ CALCULATE WIND STRESS CURL +++ ##

# GISS E2.1 SIMULATIONS
curl_e2pt1={ 'ctrl':{}, '1sEnv.ca-sa':{}, '1sEnv.ca-only':{} }
for key in curl_e2pt1.keys():
    dat = wind_stress_curl(taux_e2pt1[key], tauy_e2pt1[key])/1000 # convert g to kg
    curl_e2pt1[key] = xr.DataArray(data=dat,
                                   dims=['lat','lon'],
                                   coords=dict(lat=taux_e2pt1[key].lat,lon=taux_e2pt1[key].lon,),
                                   attrs=dict(description='Wind Stress Curl from taux and tauy',
                                              source=f'E2.1 {key}',),)
    
# MERRA-2 REANALYSIS
dat = wind_stress_curl(taux_merra, tauy_merra)
curl_merra = xr.DataArray(data=dat,
                          dims=['lat','lon'],
                          coords=dict(lat=taux_merra.lat,lon=taux_merra.lon,),
                          attrs=dict(description='Wind Stress Curl from tau_x and tau_y',
                                     source='MERRA-2',),)

# GFDL CM2.5-FLOR SIMULATIONS
curl_flor={ 'ctrl':{}, 'hitopo':{} }
for key in curl_flor.keys():
    dat = wind_stress_curl(taux_flor[key], tauy_flor[key])
    curl_flor[key] = xr.DataArray(data=dat,
                                  dims=['lat','lon'],
                                  coords=dict(lat=curl_flor[key].lat, lon=curl_flor[key].lon,),
                                  attrs=dict(description='Wind Stress Curl from tau_x and tau_y',
                                             units='N m^-3',
                                             source='FLOR CTRL',),)


In [ ]:
## +++ CALCULATE MERIDIONAL DERIVATIVE OF ZONAL INTEGRAL +++ ###
# pacific bounds: 170°W--110°W & 5°S--20°N
xmin=-170; xmax=-110
ymin=-5;   ymax=20

# NASA GISS E2.1 
dcurl_e2pt1 = { 'ctrl':{}, '1sEnv.ca-sa':{}, '1sEnv.ca-only':{} }
for key in dcurl_e2pt1.keys():
    dcurl_e2pt1[key] = curl_e2pt1.sel(lon=slice(xmin,xmax), lat=slice(ymin,ymax)).integrate('lon').differentiate('lat')

# MERRA-2    
dcurl_merra = curl_merra.sel(lon=slice(xmin,xmax), lat=slice(ymin,ymax)).integrate('lon').differentiate('lat')

# GFDL CM2.5-FLOR
dcurl_flor = { 'ctrl':{}, 'hitopo':{} }
for key in dcurl_flor.keys():
    dcurl_flor[key] = curl_flor.sel(lon=slice(xmin,xmax), lat=slice(ymin,ymax)).integrate('lon').differentiate('lat')


**plot results**

In [ ]:
tkw = {'axis': 'both', 'direction':'in', 'labelsize': 8, 'labelcolor': 'k'}   
text_kw={'color':'k', 'weight':'normal', 'size':8, 'ha':'center'}
text_kw2={'color':'k', 'weight':'bold', 'size':10, 'ha':'left'}
legend_kw={'fontsize':7, 'ncols':1,  'loc':'upper left', 'bbox_to_anchor':(0, 1.0), 'labelcolor':'linecolor', 'frameon':False}
line_kw={'lw':1} 
line_stys=['-','--',':']
flor_labels = ['CM2.5-FLOR CTRL', 'CM2.5-FLOR MAX']
e2pt1_labels = ['E2.1 CTRL', 'E2.1 CA+SA', 'E2.1 CA-ONLY']

fig=plt.figure(figsize=(6,6))

ax=plt.subplot2grid((1, 2), (0, 0), colspan=1, rowspan=1)
ax.text(-0.65, 15.1, 'U-CURRENT', **text_kw2)
ax.axvline(x=0, c='k', ls='-', lw=0.5)
# oscar obs
ax.plot(uocn_obs.mean(dim='lon'), uocn_obs.lat, c='k', ls='-', label='OBS', **line_kw)
# flor sims
for i,key in enumerate(uocn_flor.keys()):
    ax.plot(uocn_flor.mean(dim='lon'), uocn_flor.lat,
            c='r', ls=line_stys[i], label=flor_labels[i], **line_kw)
# e2.1 sims
for i,key in enumerate(uocn_e2pt1.keys()):
    ax.plot(uocn_e2pt1.mean(dim='lon'), uocn_e2pt1.lat,
            c='b', ls=line_stys[i], label=e2pt1_labels[i], **line_kw)
# set axis info
ax.set(xlim=[-0.65, 0.2], ylim=[0,15])
ax.set_xlabel('[cm/s]', **text_kw)
ax.set_ylabel('LATITUDE [°N]', **text_kw)
ax.tick_params(**tkw)
ax.legend(**legend_kw) 


ax=plt.subplot2grid((1, 2), (0, 1), colspan=1, rowspan=1)
ax.text(-12, 15.1, '$\delta$ WIND STRESS CURL', **text_kw2)
ax.axvline(x=0, c='k', ls='-', lw=0.5)
# oscar obs
ax.plot(dcurl_merra*1e7, dcurl_merra.lat, c='k', ls='-', label='MERRA-2', **line_kw)
# flor sims
for i,key in enumerate(uocn_flor.keys()):
    ax.plot(dcurl_flor*1e7, dcurl_flor.lat,
            c='r', ls=line_stys[i], label=flor_labels[i], **line_kw)
# e2.1 sims
for i,key in enumerate(uocn_e2pt1.keys()):
    ax.plot(dcurl_e2pt1*1e7, dcurl_e2pt1.lat,
            c='b', ls=line_stys[i], label=e2pt1_labels[i], **line_kw)
# set axis info
ax.set(xlim=[-12, 15], ylim=[0,15])
ax.set_xlabel('[N m$^{-3}$]', **text_kw)
ax.tick_params(**tkw)
